In [0]:
%pip install xgboost

In [0]:
# Importing libraries
import numpy as np
import pandas as pd
from scipy import stats
from pyspark.sql.functions import (
    col, count, avg, stddev, when, lit,
    current_timestamp, date_trunc, percentile_approx
)
from pyspark.sql.types import DoubleType
from pyspark.ml.functions import vector_to_array
import mlflow

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()


# Scoring and saving predictions. This simulates a production scoring table that would be updated daily

import xgboost as xgb
from sklearn.preprocessing import StandardScaler

def score_and_save(train_table, test_table, feature_names,
                   output_table, dataset_name):
    """
    Score test set and write fraud_score + metadata to Delta.
    In production this runs on new incoming data daily.
    """
    train_df = spark.table(train_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))
    test_df  = spark.table(test_table) \
                    .withColumn("is_fraud", col("is_fraud").cast("double"))

    def extract(df):
        arr = df.withColumn("features_arr", vector_to_array("features"))
        pdf = arr.select("features_arr", "is_fraud").toPandas()
        return (np.array(pdf["features_arr"].tolist()),
                pdf["is_fraud"].values)

    X_train, y_train = extract(train_df)
    X_test,  y_test  = extract(test_df)

    scaler = StandardScaler()
    scaler.fit(X_train[y_train == 0])
    X_train_sc = scaler.transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    fraud_count      = int(y_train.sum())
    legit_count      = len(y_train) - fraud_count
    scale_pos_weight = legit_count / fraud_count

    model = xgb.XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="aucpr", random_state=42, early_stopping_rounds = 30, n_jobs=-1
    )
    model.fit(X_train_sc, y_train,
              eval_set=[(X_test_sc, y_test)],
              verbose=False)

    scores      = model.predict_proba(X_test_sc)[:, 1]
    predictions = (scores >= 0.5).astype(int)

    # Build scored DataFrame
    scored_pdf = pd.DataFrame({
        "fraud_score"    : scores,
        "predicted_fraud": predictions,
        "actual_fraud"   : y_test,
        "correct"        : (predictions == y_test).astype(int),
        "dataset"        : dataset_name,
        "scored_at"      : pd.Timestamp.now()
    })

    # Add risk tier
    scored_pdf["risk_tier"] = pd.cut(
        scored_pdf["fraud_score"],
        bins  = [0, 0.3, 0.6, 0.8, 1.0],
        labels= ["low", "medium", "high", "critical"]
    ).astype(str)

    # Save to Delta
    spark.createDataFrame(scored_pdf).write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.ml_layer.{output_table}")

    print(f"Scored {len(scored_pdf):,} rows → {output_table}")
    return model, scaler, X_train_sc, X_test_sc, y_train, y_test, scores


print("Scoring applications...")
(xgb_app, scaler_app,
 X_app_train_sc, X_app_test_sc,
 y_app_train, y_app_test,
 app_scores) = score_and_save(
    "workspace.ml_layer.application_train_features",
    "workspace.ml_layer.application_test_features",
    APP_FEATURE_NAMES := [
        "log_income", "address_stability", "under_25",
        "name_email_similarity", "days_since_request",
        "zip_count_4w", "payment_type_index"
    ],
    "scored_applications", "applications"
)

print("Scoring transactions...")
(xgb_txn, scaler_txn,
 X_txn_train_sc, X_txn_test_sc,
 y_txn_train, y_txn_test,
 txn_scores) = score_and_save(
    "workspace.ml_layer.transaction_train_features",
    "workspace.ml_layer.transaction_test_features",
    TXN_FEATURE_NAMES := [
        "log_amount", "amount_balance_ratio", "transaction_hour",
        "transaction_dayofweek", "merchant_fraud_rate",
        "customer_avg_transaction_amount", "merchant_category_index",
        "device_type_index", "channel_grouped_index",
        "location_city_grouped_index"
    ],
    "scored_transactions", "transactions"
)

# Performance metrics table

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix
)

def compute_performance_metrics(y_true, y_proba, dataset_name,
                                 threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    return {
        "dataset"          : dataset_name,
        "roc_auc"          : round(roc_auc_score(y_true, y_proba), 4),
        "pr_auc"           : round(average_precision_score(y_true, y_proba), 4),
        "precision"        : round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall"           : round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1"               : round(f1_score(y_true, y_pred, zero_division=0), 4),
        "threshold"        : threshold,
        "true_positives"   : int(tp),
        "false_positives"  : int(fp),
        "true_negatives"   : int(tn),
        "false_negatives"  : int(fn),
        "fraud_rate_actual": round(float(y_true.mean()), 4),
        "fraud_rate_pred"  : round(float(y_pred.mean()), 4),
        "evaluated_at"     : pd.Timestamp.now().isoformat()
    }

metrics_app = compute_performance_metrics(
    y_app_test, app_scores, "applications", threshold=0.8067
)
metrics_txn = compute_performance_metrics(
    y_txn_test, txn_scores, "transactions", threshold=0.9380
)

metrics_df = pd.DataFrame([metrics_app, metrics_txn])
display(metrics_df)

spark.createDataFrame(metrics_df).write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ml_layer.model_performance_metrics")

print("Performance metrics saved")

# Drift Detection analysis
# PSI (Population Stability Index)
# PSI < 0.1  → no drift
# PSI 0.1–0.2 → moderate drift, monitor
# PSI > 0.2  → significant drift, retrain

def compute_psi(expected, actual, buckets=10):
    """
    Population Stability Index between training and test distributions.
    expected = training feature values
    actual   = test/production feature values
    """
    # Create bins from expected distribution
    breakpoints = np.nanpercentile(expected,
                                   np.linspace(0, 100, buckets + 1))
    breakpoints  = np.unique(breakpoints)

    expected_counts = np.histogram(expected, bins=breakpoints)[0]
    actual_counts   = np.histogram(actual,   bins=breakpoints)[0]

    # Avoid zero counts
    expected_pct = np.where(expected_counts == 0, 0.0001,
                            expected_counts / len(expected))
    actual_pct   = np.where(actual_counts == 0, 0.0001,
                            actual_counts / len(actual))

    psi = np.sum((actual_pct - expected_pct) *
                 np.log(actual_pct / expected_pct))
    return float(psi)


def run_drift_detection(X_train, X_test, feature_names, dataset_name):
    """Compute PSI for all features and classify drift severity."""
    drift_records = []
    
    # Match feature names to actual array shape
    actual_num_features = X_train.shape[1]
    feature_names_adjusted = feature_names[:actual_num_features]

    for i, feat in enumerate(feature_names_adjusted):
        psi = compute_psi(X_train[:, i], X_test[:, i])

        if psi < 0.1:
            status = "✅ Stable"
            action = "None"
        elif psi < 0.2:
            status = "⚠️  Moderate drift"
            action = "Monitor closely"
        else:
            status = "🔴 Significant drift"
            action = "Consider retraining"

        drift_records.append({
            "feature"   : feat,
            "psi"       : round(psi, 4),
            "status"    : status,
            "action"    : action,
            "dataset"   : dataset_name,
            "checked_at": pd.Timestamp.now().isoformat()
        })

    drift_df = pd.DataFrame(drift_records).sort_values(
        "psi", ascending=False
    ).reset_index(drop=True)

    print(f"\n{'='*60}")
    print(f"  DRIFT DETECTION — {dataset_name}")
    print(f"{'='*60}")
    display(drift_df)

    return drift_df


drift_app = run_drift_detection(
    X_app_train_sc, X_app_test_sc,
    APP_FEATURE_NAMES, "applications"
)
drift_txn = run_drift_detection(
    X_txn_train_sc, X_txn_test_sc,
    TXN_FEATURE_NAMES, "transactions"
)

drift_all = pd.concat([drift_app, drift_txn], ignore_index=True)
spark.createDataFrame(drift_all).write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ml_layer.drift_detection_results")

print("Drift detection results saved")

# Retraining requirements assessment

def assess_retraining_requirements(metrics_df, drift_df, dataset_name):
    """
    Combines performance degradation + drift signals into
    a retraining recommendation.
    """
    m = metrics_df[metrics_df["dataset"] == dataset_name].iloc[0]

    # Thresholds 
    PR_AUC_THRESHOLD   = 0.30   # below this → performance concern
    RECALL_THRESHOLD   = 0.40   # catching less than 40% of fraud → concern
    PSI_ALERT          = 0.20   # any feature PSI above this → drift concern

    dataset_drift = drift_df[drift_df["dataset"] == dataset_name]
    drifted_features = dataset_drift[
        dataset_drift["psi"] > PSI_ALERT
    ]["feature"].tolist()

    triggers = []

    if m["pr_auc"] < PR_AUC_THRESHOLD:
        triggers.append(
            f"PR-AUC={m['pr_auc']:.4f} below threshold {PR_AUC_THRESHOLD}"
        )
    if m["recall"] < RECALL_THRESHOLD:
        triggers.append(
            f"Recall={m['recall']:.4f} below threshold {RECALL_THRESHOLD}"
        )
    if drifted_features:
        triggers.append(
            f"Feature drift detected: {', '.join(drifted_features)}"
        )

    recommendation = "🔴 RETRAIN" if triggers else "🟢 NO ACTION NEEDED"

    result = {
        "dataset"        : dataset_name,
        "recommendation" : recommendation,
        "pr_auc"         : m["pr_auc"],
        "recall"         : m["recall"],
        "drifted_features": ", ".join(drifted_features) if drifted_features else "None",
        "triggers"       : " | ".join(triggers) if triggers else "None",
        "assessed_at"    : pd.Timestamp.now().isoformat()
    }

    print(f"\n{'='*60}")
    print(f"  RETRAINING ASSESSMENT — {dataset_name}")
    print(f"{'='*60}")
    print(f"  Recommendation  : {recommendation}")
    print(f"  PR-AUC          : {m['pr_auc']:.4f}")
    print(f"  Recall          : {m['recall']:.4f}")
    print(f"  Drifted features: {result['drifted_features']}")
    if triggers:
        print(f"  Triggers:")
        for t in triggers:
            print(f"    → {t}")

    return result


retraining_app = assess_retraining_requirements(
    metrics_df, drift_all, "applications"
)
retraining_txn = assess_retraining_requirements(
    metrics_df, drift_all, "transactions"
)

retraining_df = pd.DataFrame([retraining_app, retraining_txn])
spark.createDataFrame(retraining_df).write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ml_layer.retraining_requirements")

print("\nRetraining requirements saved")